# Notebook 01 — Data Pull and Cleaning

**DS603 Group Project** · Tan Kok Yong (01426300)

Pulls daily return data from WRDS CRSP, cleans it, and saves an analysis-ready file to `data/processed/`.

**Prerequisites:** WRDS account with CRSP access; WRDS Python package installed.

> **Data ethics:** raw CRSP extracts are saved to `data/raw/` (gitignored). Do **not** commit licensed data.

## 0. Setup

In [ ]:
# Install required packages (run once per Colab session)
!pip install -q wrds pandas numpy scipy statsmodels matplotlib seaborn pyarrow

In [ ]:
import sys, os
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/DS603_Project'
sys.path.insert(0, PROJECT_ROOT)

RAW_DIR       = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
print('Project root:', PROJECT_ROOT)

## 1. Ticker selection

We use three equities spanning different volatility profiles:

| Ticker | Name | Profile | CRSP PERMNO |
|--------|------|---------|-------------|
| JPM    | JPMorgan Chase | Stable blue-chip (financials) | 47896 |
| AAPL   | Apple Inc. | Stable blue-chip (tech) | 14593 |
| TSLA   | Tesla Inc. | High-volatility growth | 93436 |

PERMNOs verified via `crsp.stocknames`. Update if necessary.

In [ ]:
PERMNOS = {
    'JPM':  47896,
    'AAPL': 14593,
    'TSLA': 93436,
}
START_DATE = '2015-01-01'
END_DATE   = '2024-12-31'

## 2. Connect to WRDS and pull data

In [ ]:
import wrds

# wrds.Connection() reads credentials from ~/.pgpass or prompts interactively.
# In Colab, you will be prompted for your WRDS username and password.
conn = wrds.Connection()
print('Connected to WRDS.')

In [ ]:
# Verify tickers and PERMNOs via stocknames
permno_list = ', '.join(str(p) for p in PERMNOS.values())
names = conn.raw_sql(f"""
    SELECT permno, ticker, comnam, namedt, nameenddt
    FROM crsp.stocknames
    WHERE permno IN ({permno_list})
    ORDER BY permno, namedt
""")
print(names.to_string(index=False))

In [ ]:
# Pull daily stock file: date, permno, ret, prc, vol
permno_list = ', '.join(str(p) for p in PERMNOS.values())

raw = conn.raw_sql(f"""
    SELECT permno, date, ret, prc, vol
    FROM crsp.dsf
    WHERE permno IN ({permno_list})
      AND date >= '{START_DATE}'
      AND date <= '{END_DATE}'
    ORDER BY permno, date
""")

conn.close()
print(f'Pulled {len(raw):,} rows covering {raw.date.nunique()} trading days.')
raw.head()

In [ ]:
# Save raw extract locally — gitignored, never commit
raw_path = os.path.join(RAW_DIR, 'crsp_raw.parquet')
raw.to_parquet(raw_path, index=False)
print(f'Raw extract saved to {raw_path}  (gitignored — do not commit)')

## 3. Cleaning

**Cleaning decisions documented here:**
1. Drop rows with missing `RET` (non-trading days, delistings, corporate-action gaps).
2. Drop rows where `|RET| > 0.75` (>75% single-day move is almost certainly a corporate action or data error, not genuine price change).
3. Drop rows where `PRC <= 0` or is missing (price sign conventions in CRSP: negative PRC = bid–ask midpoint; we drop for consistency).
4. Require at least 252 valid trading days per ticker-year; flag any year below this threshold.
5. Forward-fill at most one missing return within a ticker-year (weekend/holiday artefacts).

In [ ]:
permno_to_ticker = {v: k for k, v in PERMNOS.items()}

df = raw.copy()
df['ticker'] = df['permno'].map(permno_to_ticker)
df['date']   = pd.to_datetime(df['date'])

n_raw = len(df)

# 1. Drop missing RET
df = df.dropna(subset=['ret'])
print(f'After dropping missing RET: {len(df):,} rows (dropped {n_raw - len(df):,})')

# 2. Drop extreme single-day moves
extreme_mask = df['ret'].abs() > 0.75
print(f'Extreme returns (|RET|>0.75): {extreme_mask.sum()} rows — inspect before dropping:')
print(df[extreme_mask][['ticker','date','ret','prc']].to_string(index=False))
df = df[~extreme_mask]

# 3. Drop non-positive price
df = df[df['prc'] > 0]

print(f'\nFinal clean rows: {len(df):,}')
print(df.groupby('ticker')['ret'].count().rename('trading_days'))

In [ ]:
# Construct the loss variable: L = -RET
df['loss'] = -df['ret']

df = df[['ticker', 'permno', 'date', 'ret', 'loss', 'prc', 'vol']].sort_values(['ticker', 'date']).reset_index(drop=True)
df.head()

## 4. Sub-period labels

| Sub-period | Window | Rationale |
|---|---|---|
| Calm | 2017-01-01 to 2019-12-31 | Pre-COVID, low-volatility bull run |
| Stressed | 2020-02-01 to 2020-12-31 | COVID crash and recovery |
| Full | entire sample | All available data |

In [ ]:
CALM_START    = pd.Timestamp('2017-01-01')
CALM_END      = pd.Timestamp('2019-12-31')
STRESSED_START = pd.Timestamp('2020-02-01')
STRESSED_END   = pd.Timestamp('2020-12-31')

def label_period(date):
    if CALM_START <= date <= CALM_END:
        return 'calm'
    elif STRESSED_START <= date <= STRESSED_END:
        return 'stressed'
    return 'other'

df['period'] = df['date'].apply(label_period)

print(df.groupby(['ticker', 'period'])['date'].count().unstack(fill_value=0))

## 5. Save processed data

In [ ]:
out_path = os.path.join(PROCESSED_DIR, 'returns_clean.parquet')
df.to_parquet(out_path, index=False)
print(f'Processed data saved: {out_path}')
print(f'Shape: {df.shape}')
df.info()

## 6. Quick sanity check

Verify data looks reasonable before moving to Notebook 02.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(len(PERMNOS), 1, figsize=(12, 9), sharex=True)
for ax, (ticker, _) in zip(axes, PERMNOS.items()):
    sub = df[df['ticker'] == ticker]
    ax.plot(sub['date'], sub['ret'], lw=0.5, color='steelblue', label=ticker)
    # Shade sub-periods
    ax.axvspan(CALM_START, CALM_END, alpha=0.15, color='green', label='Calm')
    ax.axvspan(STRESSED_START, STRESSED_END, alpha=0.15, color='red', label='Stressed')
    ax.set_ylabel('Daily Return')
    ax.set_title(ticker)
    ax.legend(loc='upper left', fontsize=7)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.suptitle('Daily Returns — Calm (green) and Stressed (red) sub-periods', y=1.01)
plt.tight_layout()
os.makedirs(os.path.join(PROJECT_ROOT, 'outputs', 'figures'), exist_ok=True)
fig.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'figures', '01_return_series.png'), dpi=150, bbox_inches='tight')
plt.show()